In [ ]:
import re
import pickle
import pandas as pd
import numpy as np
import csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
%cd "your_drive_path"

In [ ]:
!nvidia-smi

Fri Dec  1 10:03:52 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   62C    P0    31W /  70W |   1529MiB / 15360MiB |      4%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [6]:
!pip install --pre --force-reinstall mlc-ai-nightly-cu118 mlc-chat-nightly-cu118 -f https://mlc.ai/wheels


Looking in links: https://mlc.ai/wheels


ERROR: Could not find a version that satisfies the requirement mlc-ai-nightly-cu118 (from versions: none)
ERROR: No matching distribution found for mlc-ai-nightly-cu118


In [ ]:
import os
os.kill(os.getpid(), 9)

In [4]:
!mkdir -p dist/prebuilt
!git clone https://github.com/mlc-ai/binary-mlc-llm-libs.git dist/prebuilt/lib
!cd dist/prebuilt && git clone https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

Sintassi del comando errata.


Generating prompts

In [ ]:
with open("Step1-2_predicted_requirements.txt", "r") as o:
    final_predicted_rqs = o.readlines()


In [ ]:
prompts = []
for sentence in final_predicted_rqs:
  prompts.append({"prompt":"Given the following excerpt of a conversation, derive the system requirements: '"+sentence+", if any'. Please answer only with the list of derived requirements as output as shown in the following examples. Please do not consider the example excerpts provided in the examples in your final answer. ---- Example 1: ---- Example excerpt: 'Excerpt of conversation containing system requirements' Example output: '1. The system must have example feature X; 2. The system must have example feature Y;' ---- Example 2: ---- Example excerpt: 'Excerpt of conversation that does not contain system requirements' Example output: 'None'", "sentence":sentence})

In [5]:
from mlc_chat import ChatModule
from mlc_chat.callback import StreamToStdout

cm = ChatModule(model="Llama-2-7b-chat-hf-q4f16_1")

output = []
for prompt in prompts:
  output.append({"sentence":prompt["sentence"],
                "result":cm.generate(prompt=prompt["prompt"],
                                     progress_callback=StreamToStdout(callback_interval=2),
                )})
  cm.reset_chat()


ModuleNotFoundError: No module named 'mlc_chat'

In [ ]:
import json

f_output = json.dumps(output, indent=4)

print(f_output)


[
    {
        "sentence": "we're doing I'm *** and this is **** we are colleagues and uh we are here for the IFA system and all the requirements that we'll now be discussing with you so that we can deliver a good system to you. Yeah. Um Uh just just can you give us 10 seconds? We just need to start recording because. Okay. Okay. So yes. Now the video and our conversation is getting recorded. All right. So we understand that IFA's new system wants to have some key features. For example, you need a proper budget, budgetting portal, reporting portal, scheduling portal and a fan portal for your consumers, fans, users. So we'll go one by one, uh and we'll ask you some questions regarding each of these key categories and then um maybe you can add whatever we have missed and we'll just round it up nicely. So uh regarding the budget, uh We want to know the basics first. So uh how is the IFA Financed and what are your main expense categories? You would say? Expense category. So the budget con

In [ ]:

final_requirements_mapping = []

for row in output:
  results = row["result"]
  splitted = results.split("\n")
  requirements = []
  for line in splitted:
    if "The system must" in line:
      requirements.append(line)
  final_requirements_mapping.append({"sentence": row["sentence"], "requirements":requirements})

print(final_requirements_mapping)

[{'sentence': "we're doing I'm *** and this is **** we are colleagues and uh we are here for the IFA system and all the requirements that we'll now be discussing with you so that we can deliver a good system to you. Yeah. Um Uh just just can you give us 10 seconds? We just need to start recording because. Okay. Okay. So yes. Now the video and our conversation is getting recorded. All right. So we understand that IFA's new system wants to have some key features. For example, you need a proper budget, budgetting portal, reporting portal, scheduling portal and a fan portal for your consumers, fans, users. So we'll go one by one, uh and we'll ask you some questions regarding each of these key categories and then um maybe you can add whatever we have missed and we'll just round it up nicely. So uh regarding the budget, uh We want to know the basics first. So uh how is the IFA Financed and what are your main expense categories? You would say? Expense category. So the budget consists of incom